In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import sympy as sp
from IPython.display import HTML

# **Part 1: The Wave Equation**
We solve the wave equation
$$
\partial_t^2 u - c^2 \partial_x^2 u = f(t,x), \\
u(0,x) = u^0(x), \\
\partial_t u(0,x) = v^0(x)
$$
with periodic boundary conditions
$$
u(-L) = u(L).
$$



### Question 1
Implement the finite difference scheme above for the wave equation with periodic boundary conditions.

In [ ]:
def solve_wave_equation():
  L = 10
  a = -L
  b = L
  c = 1
  T = L/(2*c)
  M = 200

  h = (b - a) / M # Grid spacing for M+1 points from a to b

  N = int(c * T / h) # Number of time steps
  tau = T / N # Time step size

  xs = [a + i * h for i in range(M + 1)] # M+1 spatial points from a to b
  ts = [n * tau for n in range(N + 1)] # N+1 time points from 0 to T

  u_0 = lambda x: np.exp(-(x**2))
  v_0 = lambda x: 2*c*x*np.exp(-(x**2))
  

  U = np.zeros((M + 1, N + 1))
  f = np.zeros((M + 1, N + 1))

  # Set initial conditions at t=0 (n=0)
  for i in range(M + 1):
      U[i, 0] = u_0(xs[i])

  # Set initial conditions at t=1 (n=1) using v_0
  # For interior points (i=1 to M-1)
  for i in range(1, M):
      U[i, 1] = U[i, 0] + tau * v_0(xs[i]) + ((c**2 * tau**2) / (2 * h**2)) * (U[i+1, 0] - 2 * U[i, 0] + U[i-1, 0]) + ((tau**2 / 2) * f[i, 0])

  # For i=0 (left boundary, using periodic conditions: U[M,0] is left neighbor of U[0,0])
  U[0, 1] = U[0, 0] + tau * v_0(xs[0]) + ((c**2 * tau**2) / (2 * h**2)) * (U[1, 0] - 2 * U[0, 0] + U[M, 0]) + ((tau**2 / 2) * f[0, 0])
  # For i=M (right boundary, enforce periodic condition U[M,1] = U[0,1])
  U[M, 1] = U[0, 1]

  # Explicit timestepping for n from 1 to N-1 (calculating U[:, n+1])
  for n in range(1, N):
        for i in range(1, M): # Interior points
            U[i, n + 1] = 2*U[i,n] - U[i,n-1] + ((c**2 * tau**2) / (h**2)) * (U[i+1,n] - 2*U[i,n] + U[i-1,n]) + (tau**2)*f[i,n]
        # Boundary points
        # For i=0 (left boundary, using periodic conditions: U[M,n] is left neighbor of U[0,n])
        U[0, n + 1] = 2*U[0,n] - U[0,n-1] + ((c**2 * tau**2) / (h**2)) * (U[1,n] - 2*U[0,n] + U[M,n]) + (tau**2)*f[0,n]
        # For i=M (right boundary, enforce periodic condition U[M,n+1] = U[0,n+1])
        U[M, n + 1] = U[0,n+1]

  return U

: 

### Question 2
Test your code with the method of manufactured solutions.

Let $L = 10$, $c = 1$, $u(t,x) = \exp(-(x-ct)^2)$.
Compute the corresponding $f$, $u^0$, and $v^0$ such that $u$ solves the PDE.
Run until final time $T := L/(2c) = N\tau$.
        
Plot $U_i^N$ versus $u(t_N,x_i)$ at the grid points $x_i$ for the highest refinement level.

In [ ]:
def u_exact(x, t):
  c=1
  return np.exp(-(x-c*t)**2)

def test_wave_equation():
  L = 10
  a = -L
  b = L
  c = 1
  T = L/(2*c)
  M = 200

  u_0 = lambda x: np.exp(-(x**2))
  v_0 = lambda x: 2*c*x*np.exp(-(x**2))

  h = (b - a) / M # Grid spacing for M+1 points from a to b

  N = int(c * T / h) # Number of time steps
  tau = T / N # Time step size

  xs = [a + i * h for i in range(M + 1)] # M+1 spatial points from a to b
  ts = [n * tau for n in range(N + 1)] # N+1 time points from 0 to T

  U = solve_wave_equation()


 # Dynamically match both xs and ts arrays to the true shape of the U matrix
  xs = [a + i * h for i in range(U.shape[0])]
  ts = [n * tau for n in range(U.shape[1])]

    # set the initial plot
  fig, ax = plt.subplots()
  U_max = np.max(U)
  U_min = np.min(U)
  abs_U_max = np.max(np.abs(U))
  y_min = U_min - 0.1 * abs_U_max
  y_max = U_max + 0.1 * abs_U_max
  ax.set(
      ylim=[y_min, y_max], xlabel="x", ylabel="u(x,t)", title="t = {}".format(ts[0])
  )
  line = ax.plot(xs, U[:, 0])[0]

    # update the plot each frame
  def update(frame):
      line.set_ydata(U[:, frame])
      ax.set(title="t = {}".format(ts[frame]))
      return line

  # create the animation
  ani = animation.FuncAnimation(fig=fig, func=update, frames=U.shape[1], interval=30)
  plt.close()
  return HTML(ani.to_jshtml())
test_wave_equation()

### Question 3
Let
	      \begin{equation}
		      e_k := \max_{0\leq i \leq M} |U_i^N - u(t_N,x_i)|
	      \end{equation}
	      denote the discrete error at the final time for mesh refinement level $k$, where we recall that $M = 2^k$.
	      Create a log--log plot of $e_k$ versus the mesh size $h_k := 2L/M = L / 2^{k-1}$ for $k = 10,11,12$.

In [ ]:
def compute_errors():
    L = 10
    a = -L # Spatial domain from -L to L
    b = L
    c = 1
    T = L/(2*c) # Final time T = 5.0

    # Initial conditions for manufactured solution u(t,x) = exp(-(x-ct)^2)
    u_0 = lambda x: np.exp(-(x**2))
    v_0 = lambda x: 2*c*x*np.exp(-(x**2))

    # table headers
    print("h\t\tE_M")

    for M in [2**9,2**10,2**11]:
        # Call solve_wave_equation with parameters and capture both U and ts
        U = solve_wave_equation()

        # exact solution at the discrete grid points
        h_local = (b - a) / M # Grid spacing based on current M
        # Ensure xs matches U's spatial dimension, which is M+1 points
        xs = [a + i * h_local for i in range(M + 1)]

        U_exact = [u_exact(x , T) for x in xs]

        # Access U at the final time step using an integer index
        # Correctly calculate the maximum error
        E_M = max(abs(U_exact[i] - U[i, -1]) for i in range(M+1))

        # table entry
        print("{:e}\t{:e}".format(h_local, E_M))


compute_errors()

### Question 4
Make an animation of your approximate solution $U_i^n$.

In [ ]:
def make_animation():

### Question 5
Now set $f(t,x) = 0$ and $v^0 = 0$.
	      Experiment with different periodic initial conditions $u^0$.
	      Take snapshots of your simulations and create animations over the circle in 3d.

In [ ]:
def animate_over_circle():